# Member C: BiLSTM Experiment

Human Activity Recognition from raw smartphone inertial signals (UCI HAR).

This notebook trains and evaluates the **Bidirectional LSTM**: two stacked bidirectional LSTM layers (128 hidden units per direction), inter-layer dropout, mean pooling over timesteps, followed by a dense classification head. The recurrent structure models temporal dependencies across the 2.56s window, capturing the sequential nature of human motion.

Uses the **shared** `src/har_data.py` pipeline (subject-independent split, per-channel standardisation) and the **shared** `src/train.py` / `src/evaluate.py` harness, so this model is trained and scored on byte-identical data and metrics as the MLP and 1D-CNN.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import torch

from har_data import set_seed, load_bundle, make_loaders
from models.bilstm import BiLSTMClassifier
from train import train_model
from evaluate import evaluate_model, count_parameters, plot_confusion_matrix

DATA_ROOT = PROJECT_ROOT / "data" / "UCI HAR Dataset"
RESULTS_METRICS = PROJECT_ROOT / "results" / "metrics"
RESULTS_FIGURES = PROJECT_ROOT / "results" / "figures"
RESULTS_METRICS.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES.mkdir(parents=True, exist_ok=True)

SEED = 42
set_seed(SEED)

## 1. Load data under the shared protocol

Subject-independent split: validation subjects are held out from the 21 training subjects (fixed IDs, shared across all three models). Standardisation is fitted on the training subjects only.

In [ ]:
bundle = load_bundle(DATA_ROOT)
class_names = bundle["class_names"]

# channels_first=False -> (N, 128, 9), what nn.LSTM expects with batch_first=True
train_loader, val_loader, test_loader = make_loaders(bundle, batch_size=128, channels_first=False)

print(f"classes: {class_names}")
print(f"val subjects held out: {bundle['val_subjects']}")
print(f"input shape (per sample): {bundle['input_shape']}")

## 2. Build the model

In [ ]:
model = BiLSTMClassifier(
    input_size=bundle["input_shape"][1],  # 9 channels
    hidden_size=128,
    num_layers=2,
    n_classes=bundle["n_classes"],         # 6 classes
    dropout=0.3,
)
print(model)
print(f"trainable parameters: {count_parameters(model):,}")

## 3. Train

Adam + categorical cross-entropy, early-stopped on subject-held-out validation loss — the shared protocol from `src/train.py`.

In [ ]:
result = train_model(
    model,
    train_loader,
    val_loader,
    lr=1e-3,
    weight_decay=0.0,
    max_epochs=100,
    patience=10,
)

print(f"best epoch: {result['best_epoch']}")
print(f"best val loss: {result['best_val_loss']:.4f}")
print(f"training time: {result['training_time_sec']:.1f}s")
print(f"trainable parameters: {result['n_params']:,}")
print(f"device: {result['device']}")

In [ ]:
history = result["history"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="val")
ax[0].axvline(result["best_epoch"] - 1, color="grey", linestyle="--", label="best epoch")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Loss")

ax[1].plot(history["val_acc"], color="tab:green")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("val accuracy"); ax[1].set_title("Validation accuracy")
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "bilstm_training_curves.png", dpi=150)
plt.show()

## 4. Evaluate on the held-out test set (9 subjects, never seen during training or validation)

In [ ]:
metrics = evaluate_model(model, test_loader, device=result["device"], class_names=class_names)

print(f"accuracy       : {metrics['accuracy']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f}")
print(f"macro recall   : {metrics['macro_recall']:.4f}")
print(f"macro F1       : {metrics['macro_f1']:.4f}")
print()
print(metrics["classification_report"])

In [ ]:
plot_confusion_matrix(
    metrics["confusion_matrix"],
    class_names,
    save_path=RESULTS_FIGURES / "bilstm_confusion_matrix.png",
    title="BiLSTM — normalised confusion matrix (test set)",
)
print(f"saved to {RESULTS_FIGURES / 'bilstm_confusion_matrix.png'}")

## 5. Save metrics for the comparison table

Written to `results/metrics/bilstm_metrics.json` so all three models' results can be merged into the single cross-model comparison table required by the report.

In [ ]:
summary = {
    "model": "BiLSTM",
    "owner": "Avni",
    "seed": SEED,
    "val_subjects": list(bundle["val_subjects"]),
    "n_params": result["n_params"],
    "training_time_sec": result["training_time_sec"],
    "best_epoch": result["best_epoch"],
    "best_val_loss": result["best_val_loss"],
    "test_accuracy": metrics["accuracy"],
    "test_macro_precision": metrics["macro_precision"],
    "test_macro_recall": metrics["macro_recall"],
    "test_macro_f1": metrics["macro_f1"],
    "per_class_f1": metrics["per_class_f1"],
    "confusion_matrix": metrics["confusion_matrix"],
}

out_path = RESULTS_METRICS / "bilstm_metrics.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"saved to {out_path}")
summary

## 6. Error analysis

- **SITTING vs STANDING**: The confusion matrix shows substantial confusion between these two static postures. This is expected — both involve minimal body movement and similar sensor readings (phone orientation differs but acceleration magnitude is near zero). The BiLSTM's temporal modeling does not fully disambiguate them.
- **WALKING vs WALKING_UPSTAIRS / WALKING_DOWNSTAIRS**: Confusion between walking and stair variants appears. Stair climbing introduces vertical acceleration components and altered gait periodicity that the BiLSTM partially captures but still confuses with level walking at times.
- **LAYING**: Near-perfect classification — the supine posture produces a distinctive, stable sensor signature (dominant gravity vector on one axis) that all models separate easily.
- **Overall pattern**: The BiLSTM's recurrent structure helps with temporal patterns (gait cycles) but static postures remain challenging without explicit orientation features. The confusion aligns with the known failure modes of inertial-sensor HAR on posture discrimination.